# 02 — Infer driver distributions for new cases

This notebook processes every Markdown case in `data/test` against an immutable
snapshot of the catalog. It records applicability and categorical probability
distributions, writes a concise human-readable report, and emits requests to add
when the fixed basis does not cover material work.

Non-relevant drivers are retained in the JSONL audit record but omitted from the
Markdown report. Images and linked resources are intentionally ignored.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

import yaml
from openai import OpenAI
from pydantic import BaseModel, Field, model_validator


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config" / "pipeline.yaml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


ROOT = find_repo_root()
CONFIG = yaml.safe_load((ROOT / "config" / "pipeline.yaml").read_text())
MODEL = os.getenv("OPENAI_MODEL", CONFIG["openai"]["model"])
client = OpenAI()


def read_case(path: Path) -> str:
    # Images are intentionally outside the MVP scope. Ignore extracted-image appendices.
    text = path.read_text(encoding="utf-8")
    return text.split("## Extracted images", 1)[0].strip()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def append_jsonl(path: Path, value: dict) -> None:
    with path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(value, ensure_ascii=False) + "\n")


def utc_run_id(prefix: str) -> str:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return f"{prefix}_{stamp}"


In [ ]:
class ProbabilityValue(BaseModel):
    category_id: str
    probability: float = Field(ge=0.0, le=1.0)


class DriverAssessment(BaseModel):
    driver_id: str
    applicability: Literal["relevant", "not_relevant", "insufficient_information"]
    distribution: list[ProbabilityValue]
    evidence: list[str]
    assumptions: list[str]
    evidence_sufficiency: Literal["low", "medium", "high"]
    uncertainty_reason: str
    questions_to_reduce_uncertainty: list[str] = Field(max_length=3)


class AddRequest(BaseModel):
    request_type: Literal["ADD_DRIVER", "ADD_CATEGORY"]
    proposed_id: str
    name: str
    description: str
    reason: str
    evidence: list[str]
    possible_overlaps: list[str]


class InferenceResult(BaseModel):
    case_id: str
    project_summary: str
    scope_comments: list[str]
    driver_assessments: list[DriverAssessment]
    basis_coverage: Literal["complete", "gaps_found"]
    requests_to_add: list[AddRequest]
    notes: list[str]


In [ ]:
CATALOG_PATH = ROOT / CONFIG["paths"]["catalog"]
PROMPT_DIR = ROOT / CONFIG["paths"]["prompts"]
SYSTEM_PROMPT = (PROMPT_DIR / "infer_system.md").read_text(encoding="utf-8")
USER_PROMPT = (PROMPT_DIR / "infer_user.md").read_text(encoding="utf-8")

catalog_text = CATALOG_PATH.read_text(encoding="utf-8")
catalog = yaml.safe_load(catalog_text)
catalog_sha256 = sha256_text(catalog_text)


def infer_case(case_path: Path) -> InferenceResult:
    case_id = case_path.stem.split("_disdoc", 1)[0]
    user_prompt = USER_PROMPT.format(
        catalog_id=catalog["catalog_id"],
        catalog_version=catalog["catalog_version"],
        catalog_sha256=catalog_sha256,
        catalog=catalog_text,
        case_id=case_id,
        case_path=case_path.relative_to(ROOT),
        case_text=read_case(case_path),
    )
    response = client.responses.parse(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        text_format=InferenceResult,
    )
    if response.output_parsed is None:
        raise RuntimeError(f"The model did not return a parsed result for {case_id}.")
    result = response.output_parsed
    validate_result(result)
    return result


def validate_result(result: InferenceResult) -> None:
    catalog_drivers = {driver["id"]: driver for driver in catalog["drivers"]}
    assessments = {item.driver_id: item for item in result.driver_assessments}
    if set(assessments) != set(catalog_drivers):
        missing = sorted(set(catalog_drivers) - set(assessments))
        extra = sorted(set(assessments) - set(catalog_drivers))
        raise ValueError(f"Driver coverage mismatch. Missing={missing}; extra={extra}")

    for driver_id, assessment in assessments.items():
        if assessment.applicability != "relevant":
            if assessment.distribution:
                raise ValueError(f"{driver_id}: only relevant drivers may have a distribution")
            continue
        expected = {category["id"] for category in catalog_drivers[driver_id]["categories"]}
        actual = {item.category_id for item in assessment.distribution}
        if actual != expected:
            raise ValueError(f"{driver_id}: category coverage mismatch")
        total = sum(item.probability for item in assessment.distribution)
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"{driver_id}: probabilities sum to {total}, not 1")


def render_inference_report(result: InferenceResult) -> str:
    lines = [f"## {result.case_id}", "", result.project_summary, ""]
    if result.scope_comments:
        lines.extend(["### Scope comments", "", *[f"- {item}" for item in result.scope_comments], ""])

    relevant = [item for item in result.driver_assessments if item.applicability == "relevant"]
    lines.extend(["### Relevant drivers", ""])
    if not relevant:
        lines.append("No relevant catalog drivers were identified.")
    for item in relevant:
        driver = next(driver for driver in catalog["drivers"] if driver["id"] == item.driver_id)
        labels = {category["id"]: category["label"] for category in driver["categories"]}
        lines.extend([
            f"#### {driver['name']} (`{item.driver_id}`)", "",
            f"Evidence sufficiency: **{item.evidence_sufficiency}**", "",
            "| Category | Probability |", "|---|---:|",
        ])
        for value in item.distribution:
            lines.append(f"| {labels[value.category_id]} | {value.probability:.0%} |")
        lines.extend(["", "**Evidence**"])
        lines.extend([f"- “{value}”" for value in item.evidence] or ["- None."])
        lines.extend(["", "**Assumptions and uncertainty**"])
        lines.extend([f"- {value}" for value in item.assumptions] or ["- No explicit assumptions."])
        lines.append(f"- {item.uncertainty_reason}")
        if item.questions_to_reduce_uncertainty:
            lines.extend(["", "**Questions to reduce uncertainty**"])
            lines.extend([f"- {value}" for value in item.questions_to_reduce_uncertainty])
        lines.append("")

    unknown = [item.driver_id for item in result.driver_assessments if item.applicability == "insufficient_information"]
    lines.extend(["### Insufficient information", ""])
    lines.extend([f"- `{driver_id}`" for driver_id in unknown] or ["- None."])
    lines.extend(["", "### Requests to add", ""])
    if result.requests_to_add:
        for request in result.requests_to_add:
            lines.append(f"- **{request.request_type} `{request.proposed_id}`** — {request.reason}")
    else:
        lines.append("- None.")
    # Non-relevant drivers remain in JSONL but are intentionally omitted here.
    return "\n".join(lines) + "\n"


## Run inference

Run notebook 01 first so that the catalog is non-empty. This cell makes paid
OpenAI API calls and creates a timestamped output directory.

In [ ]:
if not catalog["drivers"]:
    raise ValueError("The catalog is empty. Run notebook 01 before inference.")

test_dir = ROOT / CONFIG["paths"]["test_cases"]
case_paths = sorted(test_dir.glob("*.md"))
if not case_paths:
    raise FileNotFoundError(f"No Markdown cases found in {test_dir}")

run_id = utc_run_id("inference")
run_dir = ROOT / CONFIG["paths"]["inference_artifacts"] / run_id
run_dir.mkdir(parents=True, exist_ok=False)
results_path = run_dir / "case_results.jsonl"
requests_path = run_dir / "requests_to_add.jsonl"
report_path = run_dir / "case_results.md"
report_path.write_text(f"# Inference report: {run_id}\n\n", encoding="utf-8")

for index, case_path in enumerate(case_paths, start=1):
    result = infer_case(case_path)
    record = {
        "run_id": run_id,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "case_path": str(case_path.relative_to(ROOT)),
        "case_sha256": sha256_text(read_case(case_path)),
        "model": MODEL,
        "catalog_id": catalog["catalog_id"],
        "catalog_version": catalog["catalog_version"],
        "catalog_sha256": catalog_sha256,
        "result": result.model_dump(),
    }
    append_jsonl(results_path, record)
    for request in result.requests_to_add:
        append_jsonl(requests_path, {
            "run_id": run_id,
            "case_id": result.case_id,
            "catalog_version": catalog["catalog_version"],
            "status": "pending_review",
            **request.model_dump(),
        })
    with report_path.open("a", encoding="utf-8") as report:
        report.write(render_inference_report(result) + "\n")
    print(f"[{index}/{len(case_paths)}] {result.case_id}: {len(result.requests_to_add)} request(s)")

summary = {
    "run_id": run_id,
    "model": MODEL,
    "case_count": len(case_paths),
    "catalog_id": catalog["catalog_id"],
    "catalog_version": catalog["catalog_version"],
    "catalog_sha256": catalog_sha256,
}
(run_dir / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary
